# Preprocessing RAG di Google Colab

Notebook ini membangun knowledge base RAG dari semua PDF di folder:
`/content/drive/My Drive/Colab Notebooks/ragxai/datapdf`

Output artefak model RAG akan disimpan ke:
`/content/drive/My Drive/Colab Notebooks/ragxai/model/faiss_index`


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
!pip uninstall -y langchain langchain-core langchain-text-splitters langchain-classic langgraph langgraph-prebuilt > /dev/null 2>&1
!pip install -q requests==2.32.4 "langchain-community>=0.3.0,<0.4.0" "langchain-core>=0.3.0,<0.4.0" "langchain-huggingface>=0.1.0,<0.2.0" "langchain-text-splitters>=0.3.0,<0.4.0" sentence-transformers faiss-cpu pypdf


In [6]:
from pathlib import Path

ROOT_DIR = Path('/content/drive/My Drive/Colab Notebooks/ragxai')
PDF_DIR = ROOT_DIR / 'datapdf'
MODEL_DIR = ROOT_DIR / 'model'
INDEX_DIR = MODEL_DIR / 'faiss_index'
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

print('Root project:', ROOT_DIR)
print('Folder PDF:', PDF_DIR)
print('Folder model:', MODEL_DIR)
print('Output index:', INDEX_DIR)


Root project: /content/drive/My Drive/Colab Notebooks/ragxai
Folder PDF: /content/drive/My Drive/Colab Notebooks/ragxai/datapdf
Folder model: /content/drive/My Drive/Colab Notebooks/ragxai/model
Output index: /content/drive/My Drive/Colab Notebooks/ragxai/model/faiss_index


In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


In [8]:
pdf_files = sorted(PDF_DIR.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'Tidak ada file PDF di folder: {PDF_DIR}')

print('Daftar PDF:')
for pdf_file in pdf_files:
    print('-', pdf_file.name)


Daftar PDF:
- An-End-to-End-Machine-Learning-Pipeline-for-Online-Purchase-Intention-Prediction-Using-Random-Forest-and-MLOps-Practices.pdf


In [9]:
documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    pages = loader.load()
    for page in pages:
        page.metadata['source_type'] = 'pdf'
        page.metadata['display_source'] = pdf_file.name
    documents.extend(pages)

print('Total halaman PDF:', len(documents))


Total halaman PDF: 11


In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = splitter.split_documents(documents)
print('Total chunk:', len(chunks))
print('Contoh metadata chunk pertama:', chunks[0].metadata if chunks else 'Tidak ada chunk')


Total chunk: 87
Contoh metadata chunk pertama: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-12T13:54:51+07:00', 'title': 'Angkasa', 'author': 'Anggraini Kusuma', 'keywords': 'Jurnal Ilmiah Bidang Teknologi Angkasa', 'moddate': '2026-02-12T13:54:51+07:00', 'source': '/content/drive/My Drive/Colab Notebooks/ragxai/datapdf/An-End-to-End-Machine-Learning-Pipeline-for-Online-Purchase-Intention-Prediction-Using-Random-Forest-and-MLOps-Practices.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_type': 'pdf', 'display_source': 'An-End-to-End-Machine-Learning-Pipeline-for-Online-Purchase-Intention-Prediction-Using-Random-Forest-and-MLOps-Practices.pdf'}


In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = FAISS.from_documents(chunks, embeddings)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(INDEX_DIR))

print('FAISS index berhasil disimpan di:', INDEX_DIR)
print('Artefak model RAG tersedia di folder:', MODEL_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index berhasil disimpan di: /content/drive/My Drive/Colab Notebooks/ragxai/model/faiss_index
Artefak model RAG tersedia di folder: /content/drive/My Drive/Colab Notebooks/ragxai/model


In [12]:
print('Isi folder output:')
for path in sorted(INDEX_DIR.glob('*')):
    print('-', path.name)


Isi folder output:
- index.faiss
- index.pkl
